# EDS vs Queensland validated clearing comparison

This notebook:
1. finds the Queensland `completed_checked` shapefiles
2. finds your EDS output rasters
3. matches them by `tile + start_date`
4. tests one match first
5. runs all matches
6. exports CSVs for later concatenation
7. adds confusion-matrix metrics, threshold sweeps, and product comparison

In [1]:

import re
import os
import tempfile
import datetime
from pathlib import Path

import boto3
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from rasterio.plot import show as rioshow
import numpy as np
from shapely.geometry import mapping
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 200)


## 0. Configuration

Edit only these settings first, then run the notebook from top to bottom.

In [2]:

# -------------------------------------------------------------------
# Configuration
# -------------------------------------------------------------------

MY_BUCKET = "dcceew-eds-data"
MY_PREFIX = "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/"

QLD_BUCKET = "dcceew-rs-data"
QLD_PREFIX = "dcceew_202602_run/"
QLD_TEXT_FILTER = "completed_checked"

# Product to evaluate first
# Expected options from your filenames include:
# "dll", "dlj", "vi_ndvi_dllmz", "vi_ndvi_dljmz", "raster_other"
MY_PRODUCT = "dll"

# Detection settings
RASTER_THRESHOLD = 1
MIN_PROP_ABOVE_THRESHOLD = 0.2


## Helper functions

In [3]:

def list_s3_keys(bucket, prefix=""):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    rows = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            rows.append({
                "bucket": bucket,
                "key": obj["Key"],
                "size": obj["Size"],
            })

    return pd.DataFrame(rows)


def find_keys_containing(bucket, prefix="", text="completed_checked", max_show=50):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    matches = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if text.lower() in key.lower():
                matches.append(key)

    print(f"Found {len(matches)} keys containing '{text}' under '{prefix}'\n")
    for k in matches[:max_show]:
        print(k)
    if len(matches) > max_show:
        print(f"\n... and {len(matches) - max_show} more")

    return matches


def extract_tile(key):
    m = re.search(r"(p\d{3}r\d{3})", str(key))
    return m.group(1) if m else None


def extract_date_group(value):
    value = str(value)
    m = re.search(r"(d\d{16})", value)
    if m:
        return m.group(1)

    m = re.search(r"(?<!\d)(\d{8})(?!\d)", value)
    if m:
        return m.group(1)

    return None


def extract_start_date(value):
    if pd.isna(value):
        return None

    value = str(value)

    m = re.fullmatch(r"d(\d{8})(\d{8})", value)
    if m:
        return m.group(1)

    m = re.fullmatch(r"(\d{8})", value)
    if m:
        return m.group(1)

    m = re.search(r"(?<!\d)(\d{8})(?!\d)", value)
    if m:
        return m.group(1)

    return None


def classify_my_output(key):
    kl = str(key).lower()
    if not kl.endswith(".tif"):
        return "other"
    if "vi-ndvi_dljmz" in kl:
        return "vi_ndvi_dljmz"
    if "vi-ndvi_dllmz" in kl:
        return "vi_ndvi_dllmz"
    if "_dlj_" in kl:
        return "dlj"
    if "_dll_" in kl:
        return "dll"
    return "raster_other"


def summarise_df(df, name, tile_col="tile", date_col=None):
    print(f"\n===== {name} =====")
    print("Rows:", len(df))

    if tile_col in df.columns:
        print("\nTop tiles:")
        print(df[tile_col].value_counts(dropna=False).head(20))

    if date_col and date_col in df.columns:
        print("\nTop date groups:")
        print(df[date_col].value_counts(dropna=False).head(20))


## 1. Build the Queensland validated shapefile inventory

In [4]:

completed_checked_keys = find_keys_containing(
    bucket=QLD_BUCKET,
    prefix=QLD_PREFIX,
    text=QLD_TEXT_FILTER
)

completed_checked_shps = [
    k for k in completed_checked_keys
    if k.lower().endswith(".shp")
]

qld_df = pd.DataFrame({"key_qld": completed_checked_shps})
qld_df["tile"] = qld_df["key_qld"].apply(extract_tile)
qld_df["date_group_qld"] = qld_df["key_qld"].apply(extract_date_group)
qld_df["start_date"] = qld_df["date_group_qld"].apply(extract_start_date)

summarise_df(qld_df, "Queensland validated shapefiles", tile_col="tile", date_col="date_group_qld")
display(qld_df.head(20))


Found 175 keys containing 'completed_checked' under 'dcceew_202602_run/'

dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.cpg
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.dbf
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.prj
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.sbn
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.sbx
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp
dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shx
dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2

,key_qld,tile,date_group_qld,start_date
0,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,p089r078,d2025070120260117,20250701
1,dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp,p090r077,d2025081720260201,20250817
2,dcceew_202602_run/lzolre_p090r079_d2025100420260201_dlwm6/lzolre_p090r079_d2025100420260201_dlwm6_completed_checked.shp,p090r079,d2025100420260201,20251004
3,dcceew_202602_run/lzolre_p090r086_d2025110520260201_dlwm5/lzolre_p090r086_d2025110520260201_dlwm5_completed_checked.shp,p090r086,d2025110520260201,20251105
4,dcceew_202602_run/lzolre_p091r076_d2025081620260131_dlwm6/lzolre_p091r076_d2025081620260131_dlwm6_completed_checked.shp,p091r076,d2025081620260131,20250816
5,dcceew_202602_run/lzolre_p091r077_d2025091720260131_dlwm6/lzolre_p091r077_d2025091720260131_dlwm6_completed_checked.shp,p091r077,d2025091720260131,20250917
6,dcceew_202602_run/lzolre_p092r076_d2025100220260122_dlwm5/lzolre_p092r076_d2025100220260122_dlwm5_completed_checked.shp,p092r076,d2025100220260122,20251002
7,dcceew_202602_run/lzolre_p092r077_d2025112820260123_dlwm5/lzolre_p092r077_d2025112820260123_dlwm5_completed_checked.shp,p092r077,d2025112820260123,20251128
8,dcceew_202602_run/lzolre_p092r088_d2025073120260123_dlwm5/lzolre_p092r088_d2025073120260123_dlwm5_completed_checked.shp,p092r088,d2025073120260123,20250731
9,dcceew_202602_run/lzolre_p093r075_d2025101820260122_dlwm5/lzolre_p093r075_d2025101820260122_dlwm5_completed_checked.shp,p093r075,d2025101820260122,20251018


## 2. Build your EDS output inventory

In [5]:

df_my = list_s3_keys(MY_BUCKET, MY_PREFIX)

my_outputs = df_my[
    df_my["key"].str.contains("/outputs/", case=False, na=False)
].copy()

my_outputs = my_outputs[
    my_outputs["key"].str.lower().str.endswith(".tif")
].copy()

my_outputs["tile"] = my_outputs["key"].apply(extract_tile)
my_outputs["date_group_my"] = my_outputs["key"].apply(extract_date_group)
my_outputs["start_date"] = my_outputs["date_group_my"].apply(extract_start_date)
my_outputs["product"] = my_outputs["key"].apply(classify_my_output)

summarise_df(my_outputs, "My EDS outputs", tile_col="tile", date_col="date_group_my")
display(my_outputs.head(20))
print("\nProducts found:")
print(my_outputs["product"].value_counts(dropna=False))



===== My EDS outputs =====
Rows: 32

Top tiles:
tile
p089r078    4
p089r079    4
p089r080    4
p089r081    4
p089r082    4
p090r077    4
p090r079    4
p090r086    4
Name: count, dtype: int64

Top date groups:
date_group_my
d2025060720260125    8
d2025082620260125    8
d2025070120260117    4
d2025081720260201    4
d2025100420260201    4
d2025110520260201    4
Name: count, dtype: int64


,bucket,key,size,tile,date_group_my,start_date,product
392,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dlj_e32756.tif,21916010,p089r078,d2025070120260117,20250701,dlj
393,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif,1963425,p089r078,d2025070120260117,20250701,dll
394,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/masks/l9olre_p089r078_d2025070120260117_dlj-dlj-clear-ge80_e32756.tif,1633949,p089r078,d2025070120260117,20250701,raster_other
395,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/masks/l9olre_p089r078_d2025070120260117_dlj-dlj-strong-ge60_e32756.tif,1655037,p089r078,d2025070120260117,20250701,raster_other
840,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/l8olre_p089r079_d2025060720260125_dlj_e32756.tif,63579962,p089r079,d2025060720260125,20250607,dlj
841,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/l8olre_p089r079_d2025060720260125_dll_e32756.tif,7387353,p089r079,d2025060720260125,20250607,dll
842,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/masks/l8olre_p089r079_d2025060720260125_dlj-dlj-clear-ge80_e32756.tif,2135565,p089r079,d2025060720260125,20250607,raster_other
843,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r079/outputs/p089r079_d2025060720260125/masks/l8olre_p089r079_d2025060720260125_dlj-dlj-strong-ge60_e32756.tif,2425185,p089r079,d2025060720260125,20250607,raster_other
1252,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/p089r080_d2025060720260125/l8olre_p089r080_d2025060720260125_dlj_e32756.tif,106859574,p089r080,d2025060720260125,20250607,dlj
1253,dcceew-eds-data,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r080/outputs/p089r080_d2025060720260125/l8olre_p089r080_d2025060720260125_dll_e32756.tif,13164315,p089r080,d2025060720260125,20250607,dll



Products found:
product
raster_other    16
dlj              8
dll              8
Name: count, dtype: int64


## 2.b Inspect one tile if you need to debug matching

Set `tile_to_check` below if you need to inspect raw Queensland and EDS keys side by side.

In [6]:

tile_to_check = None  # e.g. "p089r078"

if tile_to_check:
    print("Queensland rows for tile:")
    display(
        qld_df[qld_df["tile"] == tile_to_check][["tile", "date_group_qld", "start_date", "key_qld"]]
        .sort_values(["start_date", "key_qld"])
    )

    print("\nMy EDS rows for tile:")
    display(
        my_outputs[my_outputs["tile"] == tile_to_check][["tile", "product", "date_group_my", "start_date", "key"]]
        .sort_values(["start_date", "product", "key"])
    )


## 3. Match Queensland shapefiles to your EDS outputs

In [7]:

my_eval_rasters = my_outputs[my_outputs["product"] == MY_PRODUCT].copy()

matches = qld_df.merge(
    my_eval_rasters,
    on=["tile", "start_date"],
    suffixes=("_qld", "_my")
)

print(f"Matched {len(matches)} Queensland shapefiles to '{MY_PRODUCT}' rasters")
display(matches[["tile", "start_date", "date_group_qld", "date_group_my", "key_qld", "key"]].head(50))


Matched 4 Queensland shapefiles to 'dll' rasters


,tile,start_date,date_group_qld,date_group_my,key_qld,key
0,p089r078,20250701,d2025070120260117,d2025070120260117,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif
1,p090r077,20250817,d2025081720260201,d2025081720260201,dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r077/outputs/p090r077_d2025081720260201/l8olre_p090r077_d2025081720260201_dll_e32756.tif
2,p090r079,20251004,d2025100420260201,d2025100420260201,dcceew_202602_run/lzolre_p090r079_d2025100420260201_dlwm6/lzolre_p090r079_d2025100420260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r079/outputs/p090r079_d2025100420260201/l8olre_p090r079_d2025100420260201_dll_e32756.tif
3,p090r086,20251105,d2025110520260201,d2025110520260201,dcceew_202602_run/lzolre_p090r086_d2025110520260201_dlwm5/lzolre_p090r086_d2025110520260201_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r086/outputs/p090r086_d2025110520260201/l8olre_p090r086_d2025110520260201_dll_e32755.tif


## 4. Download helpers

In [8]:

def download_s3_file(bucket, key, local_path):
    s3 = boto3.client("s3")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(bucket, key, str(local_path))


def download_shapefile_bundle(bucket, shp_key, out_dir):
    base = os.path.splitext(shp_key)[0]
    exts = [".shp", ".dbf", ".shx", ".prj", ".cpg"]

    downloaded = []
    for ext in exts:
        key = base + ext
        local_path = Path(out_dir) / Path(key).name
        try:
            download_s3_file(bucket, key, local_path)
            downloaded.append(local_path)
        except Exception:
            pass

    shp_local = Path(out_dir) / (Path(base).name + ".shp")
    return shp_local, downloaded


def download_raster(bucket, key, out_dir):
    local_path = Path(out_dir) / Path(key).name
    download_s3_file(bucket, key, local_path)
    return local_path


## 5. Compare one matched pair

This section tests only one row from `matches`. It is for debugging. It does **not** process all matches.

In [9]:

def polygon_raster_stats(raster_path, gdf, threshold=None):
    results = []

    with rasterio.open(raster_path) as src:
        nodata = src.nodata
        gdf = gdf.to_crs(src.crs)

        for idx, row in gdf.iterrows():
            geom = [mapping(row.geometry)]

            try:
                out_image, _ = mask(src, geom, crop=True)
                data = out_image[0]

                if nodata is not None:
                    data = data[data != nodata]

                data = data[np.isfinite(data)]

                if data.size == 0:
                    results.append({
                        "index": idx,
                        "pixel_count": 0,
                        "min": np.nan,
                        "mean": np.nan,
                        "max": np.nan,
                        "prop_above_threshold": np.nan,
                    })
                    continue

                result = {
                    "index": idx,
                    "pixel_count": int(data.size),
                    "min": float(np.min(data)),
                    "mean": float(np.mean(data)),
                    "max": float(np.max(data)),
                }

                if threshold is not None:
                    result["prop_above_threshold"] = float(np.mean(data >= threshold))
                else:
                    result["prop_above_threshold"] = np.nan

                results.append(result)

            except Exception:
                results.append({
                    "index": idx,
                    "pixel_count": 0,
                    "min": np.nan,
                    "mean": np.nan,
                    "max": np.nan,
                    "prop_above_threshold": np.nan,
                })

    return pd.DataFrame(results)


def find_clearing_field(gdf):
    col_map = {col.lower(): col for col in gdf.columns}

    preferred = [
        "iscleared",
        "isclearing",
        "cleared",
        "clearing",
        "esastatus",
        "status",
    ]
    for name in preferred:
        if name in col_map:
            return col_map[name]

    for lower_col, original_col in col_map.items():
        if any(token in lower_col for token in ["clear", "clearing", "esa"]):
            return original_col

    return None


def normalise_truth_value(value):
    if pd.isna(value):
        return None
    value = str(value).upper().strip()

    if value in {"Y", "YES", "TRUE", "1"}:
        return 1
    if value in {"N", "NO", "FALSE", "0"}:
        return 0

    return None


def evaluate_match(match_row, raster_threshold=1, min_prop_above_threshold=0.2):
    with tempfile.TemporaryDirectory() as tmpdir:
        shp_local, downloaded_files = download_shapefile_bundle(
            bucket=QLD_BUCKET,
            shp_key=match_row["key_qld"],
            out_dir=tmpdir
        )

        raster_local = download_raster(
            bucket=MY_BUCKET,
            key=match_row["key"],
            out_dir=tmpdir
        )

        qld_gdf = gpd.read_file(shp_local)
        clearing_field = find_clearing_field(qld_gdf)

        if clearing_field is None:
            raise ValueError(
                f"No clearing field found in shapefile. Columns were: {list(qld_gdf.columns)}"
            )

        qld_gdf["_clearing_value"] = (
            qld_gdf[clearing_field]
            .astype(str)
            .str.upper()
            .str.strip()
        )

        qld_gdf["truth"] = qld_gdf["_clearing_value"].apply(normalise_truth_value)
        qld_truth = qld_gdf[qld_gdf["truth"].notna()].copy()

        if qld_truth.empty:
            raise ValueError("No usable Y/N truth values found in shapefile.")

        stats_df = polygon_raster_stats(
            raster_local,
            qld_truth,
            threshold=raster_threshold
        )

        qld_eval = qld_truth.join(stats_df.set_index("index"))

        qld_eval["predicted_cleared"] = (
            qld_eval["prop_above_threshold"] >= min_prop_above_threshold
        )
        qld_eval["pred"] = qld_eval["predicted_cleared"].astype(int)

        valid_eval = qld_eval[qld_eval["pred"].notna() & qld_eval["truth"].notna()].copy()

        tp = int(((valid_eval["truth"] == 1) & (valid_eval["pred"] == 1)).sum())
        fn = int(((valid_eval["truth"] == 1) & (valid_eval["pred"] == 0)).sum())
        fp = int(((valid_eval["truth"] == 0) & (valid_eval["pred"] == 1)).sum())
        tn = int(((valid_eval["truth"] == 0) & (valid_eval["pred"] == 0)).sum())

        summary = {
            "tile": match_row["tile"],
            "start_date": match_row["start_date"],
            "date_group_qld": match_row.get("date_group_qld"),
            "date_group_my": match_row.get("date_group_my"),
            "key_qld": match_row.get("key_qld"),
            "key": match_row.get("key"),
            "product": match_row.get("product", MY_PRODUCT),
            "clearing_field": clearing_field,
            "total_truth_polygons": int(len(qld_truth)),
            "usable_polygons": int(len(valid_eval)),
            "validated_cleared_polygons": int((valid_eval["truth"] == 1).sum()),
            "validated_not_cleared_polygons": int((valid_eval["truth"] == 0).sum()),
            "tp": tp,
            "fn": fn,
            "fp": fp,
            "tn": tn,
            "detected_by_my_eds": tp,
            "missed_by_my_eds": fn,
        }

        summary["recall"] = tp / (tp + fn) if (tp + fn) else np.nan
        summary["precision"] = tp / (tp + fp) if (tp + fp) else np.nan
        summary["specificity"] = tn / (tn + fp) if (tn + fp) else np.nan
        summary["accuracy"] = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
        summary["f1"] = (
            2 * summary["precision"] * summary["recall"] / (summary["precision"] + summary["recall"])
            if pd.notna(summary["precision"]) and pd.notna(summary["recall"]) and (summary["precision"] + summary["recall"]) > 0
            else np.nan
        )

        return summary, qld_eval, qld_gdf


In [10]:

# Run one matched pair first
match_index = 0

if len(matches) == 0:
    raise ValueError("No matches found. Inspect the tile/date logic above first.")

single_summary, single_eval, single_full_gdf = evaluate_match(
    matches.iloc[match_index],
    raster_threshold=RASTER_THRESHOLD,
    min_prop_above_threshold=MIN_PROP_ABOVE_THRESHOLD
)

display(pd.DataFrame([single_summary]))


,tile,start_date,date_group_qld,date_group_my,key_qld,key,product,clearing_field,total_truth_polygons,usable_polygons,...,fn,fp,tn,detected_by_my_eds,missed_by_my_eds,recall,precision,specificity,accuracy,f1
0,p089r078,20250701,d2025070120260117,d2025070120260117,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif,dll,IsClearing,1,1,...,0,0,0,1,0,1.0,1.0,NaN,1.0,1.0


In [11]:

print("Total polygons in shapefile:", len(single_full_gdf))

clearing_field = single_summary["clearing_field"]
print(f"Clearing field used: {clearing_field}")

print("\nTruth breakdown:")
print(single_full_gdf[clearing_field].value_counts(dropna=False))


Total polygons in shapefile: 7
Clearing field used: IsClearing

Truth breakdown:
IsClearing
u    6
y    1
Name: count, dtype: int64


In [12]:

# Inspect the evaluated polygons for the single pair
if single_eval is not None:
    display(
        single_eval[
            [
                "_clearing_value",
                "truth",
                "pixel_count",
                "min",
                "mean",
                "max",
                "prop_above_threshold",
                "predicted_cleared",
            ]
        ].head(20)
    )


,_clearing_value,truth,pixel_count,min,mean,max,prop_above_threshold,predicted_cleared
4,Y,1.0,138,10.0,36.702899,39.0,1.0,True


## 6. Run all matched pairs

This section processes every row in `matches`, not just the first one.

In [17]:

all_summaries = []
all_eval_parts = []
all_errors = []

for i, match_row in matches.iterrows():
    try:
        summary, eval_df, full_gdf = evaluate_match(
            match_row,
            raster_threshold=RASTER_THRESHOLD,
            min_prop_above_threshold=MIN_PROP_ABOVE_THRESHOLD
        )

        all_summaries.append(summary)

        # if eval_df is not None and not eval_df.empty:
        #     eval_df = eval_df.copy()
        #     eval_df["tile"] = match_row["tile"]
        #     eval_df["start_date"] = match_row["start_date"]
        #     eval_df["date_group_qld"] = match_row.get("date_group_qld")
        #     eval_df["date_group_my"] = match_row.get("date_group_my")
        #     eval_df["key_qld"] = match_row.get("key_qld")
        #     eval_df["key"] = match_row.get("key")
        #     eval_df["product"] = match_row.get("product", MY_PRODUCT)
        #     all_eval_parts.append(eval_df)


        if eval_df is not None and not eval_df.empty:
            eval_df = eval_df.copy()
    
        # Force all geometries to a common CRS before concatenation
        if hasattr(eval_df, "to_crs") and getattr(eval_df, "crs", None) is not None:
            eval_df = eval_df.to_crs("EPSG:3577")
    
        eval_df["tile"] = match_row["tile"]
        eval_df["start_date"] = match_row["start_date"]
        eval_df["date_group_qld"] = match_row.get("date_group_qld")
        eval_df["date_group_my"] = match_row.get("date_group_my")
        eval_df["key_qld"] = match_row.get("key_qld")
        eval_df["key"] = match_row.get("key")
    
        all_eval_parts.append(eval_df)
        
    except Exception as e:
        all_errors.append({
            "match_row_index": i,
            "tile": match_row.get("tile"),
            "start_date": match_row.get("start_date"),
            "date_group_qld": match_row.get("date_group_qld"),
            "date_group_my": match_row.get("date_group_my"),
            "key_qld": match_row.get("key_qld"),
            "key": match_row.get("key"),
            "product": match_row.get("product", MY_PRODUCT),
            "error": str(e),
        })

results_df = pd.DataFrame(all_summaries)
errors_df = pd.DataFrame(all_errors)

# if all_eval_parts:
#     all_eval_df = pd.concat(all_eval_parts, ignore_index=True)
# else:
#     all_eval_df = pd.DataFrame()
if all_eval_parts:
    all_eval_df = pd.concat(all_eval_parts, ignore_index=True)

    # CSV-friendly copy without geometry
    if "geometry" in all_eval_df.columns:
        all_eval_csv_df = all_eval_df.drop(columns=["geometry"]).copy()
    else:
        all_eval_csv_df = all_eval_df.copy()
else:
    all_eval_df = pd.DataFrame()
    all_eval_csv_df = pd.DataFrame()

    
print(f"Successful evaluations: {len(results_df)}")
print(f"Errors: {len(errors_df)}")

display(results_df.sort_values(["tile", "start_date"]))
if not errors_df.empty:
    display(errors_df)


Successful evaluations: 4
Errors: 0


,tile,start_date,date_group_qld,date_group_my,key_qld,key,product,clearing_field,total_truth_polygons,usable_polygons,...,fn,fp,tn,detected_by_my_eds,missed_by_my_eds,recall,precision,specificity,accuracy,f1
0,p089r078,20250701,d2025070120260117,d2025070120260117,dcceew_202602_run/lzolre_p089r078_d2025070120260117_dlwm6/lzolre_p089r078_d2025070120260117_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p089r078/outputs/p089r078_d2025070120260117/l9olre_p089r078_d2025070120260117_dll_e32756.tif,dll,IsClearing,1,1,...,0,0,0,1,0,1.0,1.0,NaN,1.0,1.0
1,p090r077,20250817,d2025081720260201,d2025081720260201,dcceew_202602_run/lzolre_p090r077_d2025081720260201_dlwm6_completed/lzolre_p090r077_d2025081720260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r077/outputs/p090r077_d2025081720260201/l8olre_p090r077_d2025081720260201_dll_e32756.tif,dll,IsClearing,14,14,...,0,0,0,14,0,1.0,1.0,NaN,1.0,1.0
2,p090r079,20251004,d2025100420260201,d2025100420260201,dcceew_202602_run/lzolre_p090r079_d2025100420260201_dlwm6/lzolre_p090r079_d2025100420260201_dlwm6_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r079/outputs/p090r079_d2025100420260201/l8olre_p090r079_d2025100420260201_dll_e32756.tif,dll,IsClearing,10,10,...,0,0,0,10,0,1.0,1.0,NaN,1.0,1.0
3,p090r086,20251105,d2025110520260201,d2025110520260201,dcceew_202602_run/lzolre_p090r086_d2025110520260201_dlwm5/lzolre_p090r086_d2025110520260201_dlwm5_completed_checked.shp,AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/p090r086/outputs/p090r086_d2025110520260201/l8olre_p090r086_d2025110520260201_dll_e32755.tif,dll,IsClearing,17,17,...,0,0,0,17,0,1.0,1.0,NaN,1.0,1.0


In [19]:

# Overall summary across all matched pairs
if not results_df.empty:
    overall = {
        "matched_pairs": int(len(results_df)),
        "validated_cleared_polygons": int(results_df["validated_cleared_polygons"].sum()),
        "validated_not_cleared_polygons": int(results_df["validated_not_cleared_polygons"].sum()),
        "tp": int(results_df["tp"].sum()),
        "fn": int(results_df["fn"].sum()),
        "fp": int(results_df["fp"].sum()),
        "tn": int(results_df["tn"].sum()),
    }

    overall["recall"] = overall["tp"] / (overall["tp"] + overall["fn"]) if (overall["tp"] + overall["fn"]) else np.nan
    overall["precision"] = overall["tp"] / (overall["tp"] + overall["fp"]) if (overall["tp"] + overall["fp"]) else np.nan
    overall["specificity"] = overall["tn"] / (overall["tn"] + overall["fp"]) if (overall["tn"] + overall["fp"]) else np.nan
    overall["accuracy"] = (
        (overall["tp"] + overall["tn"]) /
        (overall["tp"] + overall["tn"] + overall["fp"] + overall["fn"])
        if (overall["tp"] + overall["tn"] + overall["fp"] + overall["fn"]) else np.nan
    )
    overall["f1"] = (
        2 * overall["precision"] * overall["recall"] / (overall["precision"] + overall["recall"])
        if pd.notna(overall["precision"]) and pd.notna(overall["recall"]) and (overall["precision"] + overall["recall"]) > 0
        else np.nan
    )

    display(pd.DataFrame([overall]))


,matched_pairs,validated_cleared_polygons,validated_not_cleared_polygons,tp,fn,fp,tn,recall,precision,specificity,accuracy,f1
0,4,42,0,42,0,0,0,1.0,1.0,NaN,1.0,1.0


## 7. Export results (for later concatenation)

In [20]:

output_dir = Path("validation_outputs")
output_dir.mkdir(exist_ok=True)

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

if 'results_df' in globals() and not results_df.empty:
    results_path = output_dir / f"eds_validation_summary_{timestamp}.csv"
    results_df.to_csv(results_path, index=False)
    print(f"Saved summary results: {results_path}")

if 'all_eval_df' in globals() and not all_eval_df.empty:
    eval_path = output_dir / f"eds_validation_polygon_level_{timestamp}.csv"
    all_eval_df.to_csv(eval_path, index=False)
    print(f"Saved polygon-level results: {eval_path}")

if 'errors_df' in globals() and not errors_df.empty:
    errors_path = output_dir / f"eds_validation_errors_{timestamp}.csv"
    errors_df.to_csv(errors_path, index=False)
    print(f"Saved errors: {errors_path}")

if 'overall' in globals():
    overall_path = output_dir / f"eds_validation_overall_{timestamp}.csv"
    pd.DataFrame([overall]).to_csv(overall_path, index=False)
    print(f"Saved overall summary: {overall_path}")


Saved summary results: validation_outputs/eds_validation_summary_20260423_054520.csv
Saved polygon-level results: validation_outputs/eds_validation_polygon_level_20260423_054520.csv
Saved overall summary: validation_outputs/eds_validation_overall_20260423_054520.csv


## 8. Threshold sweep

Run this after section 6 if you want to compare multiple threshold settings for the current `MY_PRODUCT`.

In [21]:

threshold_grid = [5, 10, 15, 20, 25, 30]
prop_grid = [0.05, 0.10, 0.20, 0.30]

sweep_rows = []

for raster_threshold in threshold_grid:
    for min_prop in prop_grid:
        run_summaries = []
        run_errors = []

        for _, match_row in matches.iterrows():
            try:
                summary, _, _ = evaluate_match(
                    match_row,
                    raster_threshold=raster_threshold,
                    min_prop_above_threshold=min_prop
                )
                run_summaries.append(summary)
            except Exception as e:
                run_errors.append(str(e))

        if run_summaries:
            tmp = pd.DataFrame(run_summaries)

            tp = int(tmp["tp"].sum())
            fn = int(tmp["fn"].sum())
            fp = int(tmp["fp"].sum())
            tn = int(tmp["tn"].sum())

            recall = tp / (tp + fn) if (tp + fn) else np.nan
            precision = tp / (tp + fp) if (tp + fp) else np.nan
            specificity = tn / (tn + fp) if (tn + fp) else np.nan
            accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
            f1 = (
                2 * precision * recall / (precision + recall)
                if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
                else np.nan
            )

            sweep_rows.append({
                "product": MY_PRODUCT,
                "raster_threshold": raster_threshold,
                "min_prop_above_threshold": min_prop,
                "matched_pairs": len(tmp),
                "tp": tp,
                "fn": fn,
                "fp": fp,
                "tn": tn,
                "recall": recall,
                "precision": precision,
                "specificity": specificity,
                "accuracy": accuracy,
                "f1": f1,
                "errors": len(run_errors),
            })

sweep_df = pd.DataFrame(sweep_rows).sort_values(
    ["f1", "recall", "precision"],
    ascending=[False, False, False]
)

display(sweep_df.head(20))


,product,raster_threshold,min_prop_above_threshold,matched_pairs,tp,fn,fp,tn,recall,precision,specificity,accuracy,f1,errors
0,dll,5,0.05,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
1,dll,5,0.10,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
2,dll,5,0.20,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
3,dll,5,0.30,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
4,dll,10,0.05,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
5,dll,10,0.10,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
6,dll,10,0.20,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
7,dll,10,0.30,4,42,0,0,0,1.000000,1.0,NaN,1.000000,1.000000,0
8,dll,15,0.05,4,41,1,0,0,0.976190,1.0,NaN,0.976190,0.987952,0
12,dll,20,0.05,4,41,1,0,0,0.976190,1.0,NaN,0.976190,0.987952,0


## 9. Compare products

Run this after sections 1 to 5 if you want to compare multiple raster products using the current threshold settings.

In [22]:

products_to_test = ["dll", "dlj", "vi_ndvi_dllmz", "vi_ndvi_dljmz"]

product_rows = []

for product in products_to_test:
    my_eval_rasters = my_outputs[my_outputs["product"] == product].copy()

    product_matches = qld_df.merge(
        my_eval_rasters,
        on=["tile", "start_date"],
        suffixes=("_qld", "_my")
    )

    run_summaries = []
    run_errors = []

    for _, match_row in product_matches.iterrows():
        try:
            summary, _, _ = evaluate_match(
                match_row,
                raster_threshold=RASTER_THRESHOLD,
                min_prop_above_threshold=MIN_PROP_ABOVE_THRESHOLD
            )
            run_summaries.append(summary)
        except Exception as e:
            run_errors.append(str(e))

    if run_summaries:
        tmp = pd.DataFrame(run_summaries)

        tp = int(tmp["tp"].sum())
        fn = int(tmp["fn"].sum())
        fp = int(tmp["fp"].sum())
        tn = int(tmp["tn"].sum())

        recall = tp / (tp + fn) if (tp + fn) else np.nan
        precision = tp / (tp + fp) if (tp + fp) else np.nan
        specificity = tn / (tn + fp) if (tn + fp) else np.nan
        accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) else np.nan
        f1 = (
            2 * precision * recall / (precision + recall)
            if pd.notna(precision) and pd.notna(recall) and (precision + recall) > 0
            else np.nan
        )

        product_rows.append({
            "product": product,
            "matched_pairs": len(tmp),
            "tp": tp,
            "fn": fn,
            "fp": fp,
            "tn": tn,
            "recall": recall,
            "precision": precision,
            "specificity": specificity,
            "accuracy": accuracy,
            "f1": f1,
            "errors": len(run_errors),
        })

product_compare_df = pd.DataFrame(product_rows).sort_values(
    ["f1", "recall", "precision"],
    ascending=[False, False, False]
)

display(product_compare_df)


/env/lib/python3.12/site-packages/rasterio/mask.py:191: NodataShadowWarning: The dataset's nodata attribute is shadowing the alpha band. All masks will be determined by the nodata attribute
  out_image = dataset.read(
/env/lib/python3.12/site-packages/rasterio/mask.py:191: NodataShadowWarning: The dataset's nodata attribute is shadowing the alpha band. All masks will be determined by the nodata attribute
  out_image = dataset.read(
/env/lib/python3.12/site-packages/rasterio/mask.py:191: NodataShadowWarning: The dataset's nodata attribute is shadowing the alpha band. All masks will be determined by the nodata attribute
  out_image = dataset.read(
/env/lib/python3.12/site-packages/rasterio/mask.py:191: NodataShadowWarning: The dataset's nodata attribute is shadowing the alpha band. All masks will be determined by the nodata attribute
  out_image = dataset.read(
/env/lib/python3.12/site-packages/rasterio/mask.py:191: NodataShadowWarning: The dataset's nodata attribute is shadowing the alp

,product,matched_pairs,tp,fn,fp,tn,recall,precision,specificity,accuracy,f1,errors
0,dll,4,42,0,0,0,1.0,1.0,NaN,1.0,1.0,0
1,dlj,4,42,0,0,0,1.0,1.0,NaN,1.0,1.0,0


## 10. Export threshold and product comparison tables

In [23]:

if 'sweep_df' in globals() and not sweep_df.empty:
    sweep_path = output_dir / f"eds_validation_threshold_sweep_{timestamp}.csv"
    sweep_df.to_csv(sweep_path, index=False)
    print(f"Saved threshold sweep: {sweep_path}")

if 'product_compare_df' in globals() and not product_compare_df.empty:
    product_path = output_dir / f"eds_validation_product_compare_{timestamp}.csv"
    product_compare_df.to_csv(product_path, index=False)
    print(f"Saved product comparison: {product_path}")


Saved threshold sweep: validation_outputs/eds_validation_threshold_sweep_20260423_054520.csv
Saved product comparison: validation_outputs/eds_validation_product_compare_20260423_054520.csv
